<a href="https://colab.research.google.com/github/tedteske/ndplex/blob/master/notebooks/ndplex_examples.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The n-Dimensional m-Plex ("ndplex") Array: Examples

`ndplex` is a generic Pascal's array: a rectangle, cube or higher-dimensional array whose edges and node-generating rule you choose. By default it is Pascal's rectangle, with ones on the edges and every other node the sum of its neighbours above and to the left.

These are the examples from `ndplex_array.ipynb` (Ted Teske, 2019–2024), run on the `ndplex` package from PyPI instead of the notebook's own class. The original notebook stays in the repository, unchanged, as the record of the original work.

- **What changed here.** Cells that differ from the original are followed or preceded by a **Package note** saying what changed and why. The package's own changes are listed in [CHANGELOG.md](https://github.com/tedteske/ndplex/blob/master/CHANGELOG.md).
- **Network.** The `oeis` cells look sequences up on oeis.org and need a network connection; everything else runs offline once `ndplex` is installed.
- **Plotting.** `plot_ndplex` and `diag_as_hist` are not part of the package. Working copies are in the helper cell below.

Author: Ted Teske (original examples, 2019–2024). Adapted to the package with Claude (Anthropic), 2026. MIT License. See [ATTRIBUTION.md](https://github.com/tedteske/ndplex/blob/master/ATTRIBUTION.md) for sources.

## Setup

In [ ]:
%pip install -q ndplex

In [ ]:
%matplotlib inline
from importlib.metadata import version

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from ndplex import ndplex, oeis

# The original notebook's library cell called sns.set(style="darkgrid") on
# import. The package has no import side effects, so the look is set here.
sns.set_theme(style="darkgrid")

print("ndplex", version("ndplex"), "| numpy", np.__version__)

### Plotting helpers

`plot_ndplex` draws a 2-D ndplex (or its triangle) with optional highlighted diagonals. `diag_as_hist` draws 1-D diagonals as weighted histograms, optionally with a normal curve for comparison. Both are from the original notebook's library cell, with the fixes listed at the top of the cell. They need only NumPy and matplotlib.

In [ ]:
# Plotting helpers, from the library cell of ndplex_array.ipynb (Ted Teske,
# 2019-2024). They are not part of the ndplex package (from 0.1.0, plotting
# is left to the user). Changes from the original, made with Claude in 2026:
#   - diag_as_hist: legend_handles (legendHandles was removed in matplotlib
#     3.9); colours cycle, so more than six data sets work; the normal pdf is
#     computed with NumPy instead of scipy.stats.norm.
#   - plot_ndplex: the serif font is set for this figure only (the original
#     set plt.rcParams["font.family"], which changed every later plot);
#     negative values are drawn (the original drew only values > 0; zeros
#     are still left blank, because as_tri pads with zeros); the first
#     argument is named `array` (it was `ndplex`, which hid the class).
# Both still show the figure and return None, as before.

import itertools as it

import matplotlib.pyplot as plt
import numpy as np


def _normal_pdf(x, mu, sigma):
    """Normal probability density, computed with NumPy."""
    return np.exp(-0.5 * ((x - mu) / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi))


def diag_as_hist(w, x_values=None, labels=None, normed=True):
    """
    Plots weighted histogram(s) from 1D array(s), with the weights being the
    values of the array. Optionally plots a normal PDF for comparison.

    Parameters
    ----------
    w : 1D array or list of 1D arrays
    x_values : None or list of scalars
    labels : None or list of strings
    normed : bool
        If True, the area under the histogram sums to 1, and the plot displays
        a normal PDF with the data's weighted mean and standard deviation.
    """
    fig, ax = plt.subplots(figsize=(10, 7))
    colors = ['b', 'r', 'c', 'm', 'g', 'y']

    if np.isscalar(w[0]):
        w = [w]
    for i in range(len(w)):
        color = colors[i % len(colors)]
        if x_values is None:
            x_values_ = np.arange(len(w[i]))
        else:
            x_values_ = np.asarray(x_values)
        weights = np.asarray(w[i], dtype=float)

        # Plots histogram. If normed, the area under the histogram sums to 1.
        if labels is None:
            hlabel = "Data set " + str(i)
        else:
            hlabel = labels[i]
        ax.hist(x_values_, len(x_values_), weights=weights, density=normed,
                color=color, alpha=0.4, label=hlabel)

        # Plots the normal PDF with the same mean and standard deviation.
        if normed:
            if i == 0:
                nlabel = r'$N(\mu, \sigma^2)$'
            else:
                nlabel = None
            avg = np.average(x_values_, weights=weights)
            stdev = np.average((x_values_ - avg)**2, weights=weights)**0.5
            xgrid = np.linspace(min(x_values_), max(x_values_), 100)
            ax.plot(xgrid, _normal_pdf(xgrid, avg, stdev),
                    color=color, lw=2, label=nlabel)

    leg = ax.legend()
    if normed:
        leg.legend_handles[0].set_color('k')
    plt.show()


def plot_ndplex(array, move=None, loc=None,
                as_tri=False, scale=1,
                invert_xaxis=False, invert_yaxis=True,
                title=None, caption=None,
                xlabel="column or diagonal", ylabel="row",
                fontsize=12,
                move_color='lightpink'):
    """
    Plots a 2D ndplex to a grid.

    Parameters
    ----------
    array : 2D ndplex (or ndarray)
    move : tuple of ints or floats
        direction with which diagonal cuts through the array
    loc : int
        row at which diagonal starts
    as_tri : bool|'left'|'center'
        if not False, displays the array as triangle with indicated horizontal
        alignment (default 'center')
    scale : int
        scales overall size of figure, retaining aspect ratio
    invert_xaxis : bool
        if True, inverts xaxis and xticklabels
    invert_yaxis : bool; default True
        if True, inverts yaxis but not yticklabels
    title : string
        text of title above grid
    caption : string
        text of caption below grid
    xlabel : string or None
        text of xlabel
    ylabel : string or None
        text of ylabel
    fontsize : int
        fontsize for all text elements

    Zero nodes are left blank (as_tri pads its triangle with zeros).
    """
    def halign(A, ha='left'):
        """
        Horizontally aligns point A.
        """
        if ha == 'center':
            y = A[1]
            x = -A[1]/2 + A[0] + cols/2 - 0.5
            A = (x, y)
        return A

    def intersect(A, B, C, D):
        """
        Returns the coordinates of intesection of segments AB and CD using
            y = mx + b
            x = (y - b)/m
        """
        mAB = (A[1] - B[1]) / (A[0] - B[0])
        bAB = A[1] - mAB * A[0]

        mCD = (C[1] - D[1]) / (C[0] - D[0])
        bCD = C[1] - mCD * C[0]

        x = -(bCD - bAB) / (mCD - mAB)
        y = mAB * x + bAB
        return (x, y)

    rows, cols = array.shape[0], array.shape[1]

    # The serif font applies to this figure only: rc_context restores the
    # global settings when the block ends, after the figure has been shown.
    with plt.rc_context({"font.family": "serif"}):
        figsize = tuple(scale * x for x in array.shape[::-1])
        fig, ax = plt.subplots(facecolor='lightgray', figsize=figsize)
        fig.text(.5, .02, caption, ha='center', fontsize=fontsize)
        if xlabel:
            plt.xlabel(xlabel, fontsize=fontsize)
        if ylabel:
            plt.ylabel(ylabel, fontsize=fontsize)
        plt.title(title, fontsize=fontsize)
        ax.axis([-0.5, cols-0.5, -0.5, rows-0.5])
        ax.axis(True)

        if invert_yaxis:
            plt.gca().invert_yaxis()

        if invert_xaxis:
            xticks = range(cols)
            plt.gca().invert_xaxis()
            plt.xticks(xticks, xticks[::-1])

        if as_tri:
            if as_tri != 'left':
                as_tri = 'center'
            array = array.as_tri()

        # Prints nodes onto grid (every non-zero node, negative ones too).
        for i in it.product(range(rows), range(cols)):
            if array[i] != 0:
                E = halign((i[1], i[0]), as_tri)
                ax.text(E[0], E[1],
                        array[i],
                        horizontalalignment='center',
                        verticalalignment='center',
                        fontsize=fontsize)

        # Prints diagonals through nodes from point A to point B.
        if move is not None and not (move[0] == 0 and move[1] == 0):
            if move[1] < 0:
                move = (-move[0], -move[1])

            if loc is None:
                rows_range = range(0, rows)
            else:
                rows_range = range(loc, loc + 1)

            for i in rows_range:

                A = (0, i)

                if move[0] * move[1] > 0:
                    B = ((cols - 1 - i) * move[1]/move[0], rows - 1)
                elif move[0] == 0:
                    B = (i, i)
                elif move[1] == 0:
                    B = (0, rows - 1)
                else:
                    B = (-i * move[1]/move[0], 0)

                try:
                    if move[0] < move[1]:

                        if as_tri:
                            C = (0, 0)
                        else:
                            C = (cols-1, 0)

                        D = (cols-1.01, rows-1)
                        B = intersect(A, B, C, D)

                        if B[1] > rows - 1:
                            C = (0, rows-1)
                            B = intersect(A, B, C, D)

                except ZeroDivisionError:
                    pass

                A = halign(A, as_tri)
                B = halign(B, as_tri)
                col, row = [A[0], B[0]], [A[1], B[1]]
                plt.plot(col, row, marker='', color=move_color)

        plt.show()

## Example Usage

### First 2D (Pascal's Triangle and Rectangle) Examples

Let's start with a standard Pascal's rectangle, which is a Pascal's triangle rotated counter-clockwise 45 degrees.

In [ ]:
p = ndplex(9)
p

The  counting numbers are in row 1. Note that row and column numbers start with 0.

In [ ]:
p[1]

also in column 1

In [ ]:
p[:,1]

alternately, using the `diag` function

In [ ]:
p.diag(loc=(1,0),move=(0,1))

The triangular numbers are in row 2 and column 2. See https://oeis.org/A000217

In [ ]:
column = 2
p.diag(loc=(0,column),move=(1,0))

You can search all rows in OEIS with the `oeis()` function.

In [ ]:
# Needs the network: looks the rows up on oeis.org.
oeis(p)

Print diagonal 4.

In [ ]:
diagonal = 4
p.diag(loc=(diagonal,0),move=(-1,1))

The `diag` function without arguments returns the main diagonal.

In [ ]:
p.diag()

Use the `diag` function to display as a triangle, justified left.

In [ ]:
p.diag(move=[(-1,1),(1,0)])

Printing the triangle without the zeros

In [ ]:
triangle = p.diag(move=[(-1,1),(1,0)])
for row in triangle:
    print(np.trim_zeros(row).tolist())

The diagonal (row in the above triangle) sums are the powers of 2.

In [ ]:
p.diag(move=[(-1,1),(1,0)],sum_axis=1)

We can also use the `as_tri` function to do the same thing.

In [ ]:
p.as_tri()

In [ ]:
p.as_tri(sums=True)

In [ ]:
for row in p.as_tri():
    print(np.trim_zeros(row).tolist())

### 2D Plotting Examples

Standard Pascal's rectangle, highlighting the counting numbers on row 1.

In [ ]:
plot_ndplex(p, move=(0,1), loc=1, title='Pascal\'s Rectangle', caption='Counting numbers are in row 1.')

highlighting the main diagonal.

In [ ]:
plot_ndplex(p, move=(1,1), loc=0)

Pascal's Triangle, aligned center:

In [ ]:
plot_ndplex(p, as_tri='center', title='Pascal\'s Triangle')

The triangular numbers are in diagonal 2.

In [ ]:
plot_ndplex(p, move=(1,1), loc=2, as_tri='center', invert_xaxis=True,
            title='Pascal\'s Triangle',
            caption="The triangular numbers are in diagonal 2.", xlabel=None)

highlighting rows and displaying sums in the caption:

In [ ]:
caption = "Row sums = " + str(p.diag(move=[(-1,1),(1,0)],sum_axis=1))
plot_ndplex(p, move=(0,1), as_tri='center', caption=caption)

highlighting the same nodes, but arranged as a rectangle:

In [ ]:
caption = "Diagonal sums = " + str(p.diag(move=[(-1,1),(1,0)],sum_axis=1))
plot_ndplex(p, move=(-1,1), as_tri=False, caption=caption, ylabel = "row and diagonal", xlabel="column")

### First 3D (Pascal's Tetrahedron) Examples

Initialize a 3D array and show the array. Think of each group as a horizontal layer of a sandwich. The first layer, layer 0, is the top slice of bread.

In [ ]:
p3 = ndplex(5,edges=3)
p3

The second row (row 1) of the second group can be extracted using array slicing:

In [ ]:
p3[1,1,:]

or this way, using the `diag` function. These are the oblong numbers:  https://oeis.org/A002378

In [ ]:
p3.diag(loc=(1,1,0), move=(0,0,1))

`diag` without arguments returns the main diagonal:

In [ ]:
p3.diag()

Return  layer 0 row 0 [0,0,:], then layer 1 row 1 [1,1,:], then layer 2 row 2 [2,2,:], etc

In [ ]:
p3.diag(loc=(0,0), move=(1,1))

Return a diagonal cross-section starting at the upper left corner:

In [ ]:
p3.diag(loc=(0,0,0), move=[(0,-1,1),(1,1,0)])

Column 0 are https://oeis.org/A000984 and column 1 are https://oeis.org/A005430

Display a horizontal cross-section through the Pascal's tetrahedron at row 4.

In [ ]:
row = 4
p3.diag(loc=(0,row,0),move=[(0,-1,1),(1,-1,0)])

To display the set of horizontal cross-sections up to and including row 4, use the `as_tri` function.

In [ ]:
p3.as_tri(0,4)

Since the row sums of a standard Pascal's triangle are the powers of 2, one would expect that layer sums of a Pascal's tetrahedron are the powers of 3:

In [ ]:
p3.as_tri(0,4,sums=True)

### The `edges` Parameter

Let's start with placing the counting numbers on the edges of a 2D ndplex.

In [ ]:
size = 9
start = 1
#c = np.linspace(1,size,size) #allows float steps
c = np.arange(start,size+start)
ndplex(size,edges=(c,c))

Row 2 are the "egg-drop numbers" https://oeis.org/A055795

Next, put the counting numbers on one edge and negatives on the other. Start at 0. Row 2 are https://oeis.org/A254749

In [ ]:
c = np.arange(size)
ndplex(size,edges=(c,-c))

In [ ]:
# Needs the network: looks the rows up on oeis.org.
oeis(ndplex(size,edges=(c,-c)))

Put 0 on the first row of a 3D array. Notice the squares in layer 1, column 1.

In [ ]:
ndplex(5,edges=(1,1,0))[0:2]

#### Odds and Evens

In [ ]:
def odds(size):
    return np.arange(1,size+1)*2-1

def evens(size):
    return np.arange(1,size+1)*2

What if we alter the standard rectangle by placing a 2 in row 0? Notice that row 1 is then the odd numbers and row 2 is the squares.

In [ ]:
ndplex(9, edges=(1,2))

Starting with the odds in row 0:

In [ ]:
ndplex(9, edges=(1,odds(9)))

The third row contains the square pyramidal numbers: https://en.wikipedia.org/wiki/Square_pyramidal_number

The fourth row contains the 4-dimensional square pyramidal numbers: https://oeis.org/A002415

In [ ]:
# Needs the network: looks the rows up on oeis.org.
oeis(ndplex(9, edges=(1,odds(9))))

Now, let's put the odds on one edge and evens on the other. Start both series at 0 so they share a common origin.

In [ ]:
odds0 = [0] + list(odds(size-1))
evens0 = [0] + list(evens(size-1))
ndplex(9, edges=(evens0,odds0))

The second row (row 1) is are the numbers generated by $2^n + 2$ , which can be found here: https://oeis.org/A059100  
At least for the first 9 numbers, multiplying two adjacent numbers yields a number that is also in this sequence.

### The `func` and `arg_locs` Parameters

By default,  each node of a `ndplex` is the sum of the immediately previous nodes along each dimension. This is known as Pascal's rule. For example, in the standard array below,

`the 2 at (row 1, col 1) = the 1 at (row 0, col 1) + the 1 at (row 1, col 0)`

The default generating function can be expressed as a summary statistic like this:

In [ ]:
f = lambda x:sum(x)
ndplex(5, func=f)

or by adding the nodes like this:

In [ ]:
f = lambda x: x[0] + x[1]
ndplex(5, func=f)

which allows you to treat each node separately.

In [ ]:
f = lambda x: x[0] + x[1] * 2
ndplex(5, func=f)

The generating function does not have to be a one-liner:

In [ ]:
def f(x):
    a = x[0]
    b = x[1]
    return (a**2 + b**2)**0.5 #Pythagorean Theorem

ndplex(5,func=f, dtype=float).round(3)

Row 1 are the square roots of the counting numbers.

The `arg_locs` parameter allows the user to specify which nodes, relative to the subject node, will supply input values to the node-generating function. In the default case below,  each node is just the sum of the nodes immediately above and to the left. For example,

`3 at node [2,1] = x[0] + x[1] = 2 at [1,1] + 1 at [2,0]`

In [ ]:
f = lambda x: x[0] + x[1]
x = [[-1,0],[0,-1]]
ndplex(5, arg_locs=x, func=f)

Notice that the parameters to `func f` correspond to the  relative coordinates in `arg_locs`.

Here is an example where the node-generating function sums the nodes immediately above, to the left, and upper-left diagonally.

In [ ]:
f = lambda x: x[0] + x[1] + x[2]
x = [[-1,0],[0,-1],[-1,-1]]
q = ndplex(5, arg_locs=x, func=f)
q

which yields some interesting sequences.

In [ ]:
# Needs the network: looks the rows up on oeis.org.
oeis(q)

If a relative node has a coordinate set to `None`, then that coordinate is set to 0 (i.e. on an edge). For example, this is the multiplication table.

In [ ]:
start = 1
size = start + 9
c = np.arange(start, size + start)
f = lambda x: x[0] * x[1]
x = [[None, 0], [0, None]]
ndplex(size, edges=(c, c), arg_locs=x, func=f)

Here is the concave function
$z = 100 -x^2 - y^2 + $
where $z$ is displayed as an array of values, rather than a 3D surface.

In [ ]:
size = 10
start = -5
c = np.arange(start, size + start)
z = lambda x: -x[0]**2 + -x[1]**2 + 100
x = [[None, 0], [0, None]]
a = ndplex(size, edges=(c, c), arg_locs=x, func=z)
a

### Visualizing1D patterns with the `diag` function

#### Graphing diagonals

Distplay a 1D diagonal as a histogram.

In [ ]:
size = 5
p = ndplex(size,edges=2)
y = p.diag(loc=(size-1,0),move=(-1,1))
diag_as_hist(y, labels=['Base of Pascal\'s Triangle'], normed=False)

In [ ]:
sizes = [8,7,6,5] #the last one is in front

y, labels = [],[]
for size in sizes:
    p = ndplex(size)
    y.append(p.diag(loc=(size-1,0),move=(-1,1)))
    labels.append("size = " + str(size))

diag_as_hist(y, labels=labels, normed=False)

#### An oscillating function

In [ ]:
size = 50
func = lambda x: x[0]-x[1]
p = ndplex(size,edges=2,dtype=float, func=func)
y = p.diag(loc=(size-1,0),move=(-1,1))
sns.lineplot(x=np.arange(len(y)), y=y)

In [ ]:
p = ndplex(9,edges=2,dtype=int, func=func)
p

Notice that row 3 alternates the squares and the standard 3D array's [1,1,:] series, also known as the oblong numbers . See https://oeis.org/A002378

In [ ]:
p[3]

#### Adjacent Diagonal Number Ratios

In [ ]:
def adj_ratios(g):
    # The first ratio is inf where a diagonal starts with 0 (two cells below);
    # it is printed, so NumPy's divide-by-zero warning is not needed.
    with np.errstate(divide="ignore"):
        ratios = g[1:]/g[:-1]
    print("first 5 ratios =", ratios[:5])
    print("last = ",ratios[-1])
    sns.lineplot(x=np.arange(len(g)-1), y=ratios)
    plt.show()

The ratio of adjacent numbers on the main diagonal of a standard Pascal's rectangle appears to converge on 4.

In [ ]:
size = 100
g = ndplex(size,edges=2,dtype=float).diag()
adj_ratios(g)

even when changing the `edges` parameter.

In [ ]:
edges = (1, 2)
g = ndplex(size,edges=edges,dtype=float).diag()
adj_ratios(g)

In [ ]:
nums = np.arange(100)

edges = (1, nums)
g = ndplex(size,edges=edges,dtype=float).diag()
adj_ratios(g)

even when the `edges` go in opposite directions

In [ ]:
nums = np.arange(100)
dnums = np.flip(nums)

edges = (dnums, nums)
g = ndplex(size,edges=edges,dtype=float).diag()
adj_ratios(g)

...but not when when changing the `func` parameter.

In [ ]:
edges = (1,1)
func = lambda x: x[0]+x[1]*2
g = ndplex(size,edges=edges,func=func, dtype=float).diag()
adj_ratios(g)

### Visualizing 2D patterns with heatmaps and modular math

#### Checking for the occurence of 1 or more items

Use a heatmap to look for the occurence of perfect squares within a standard Pascal's rectangle. Refer to the following for available color palettes:
* https://seaborn.pydata.org/tutorial/color_palettes.html
* http://colorbrewer2.org/#type=sequential&scheme=BuGn&n=3

In [ ]:
import pandas as pd
size = 20

b = ndplex(size, dtype=np.int64)

max_sqrt = np.int64(np.max(b)**0.5+1)
squares = np.arange(2,max_sqrt)**2

df = pd.DataFrame(b)
sns.heatmap(df.mask(~df.isin(squares)))

#### Modular Math and Fractals

Let's start with creating a fractal called a Sierpinki's Gasket. See here for background: http://orion.math.iastate.edu/reu/oldREU/modupasc.htm (Iowa State REU page; see also https://en.wikipedia.org/wiki/Sierpi%C5%84ski_triangle)

> **Package note.** From 0.1.0, ndplex warns when node values overflow `int64`. `ndplex(100)` overflows from row + column = 67 on, so this cell and the next build the array with `dtype=object` (exact Python integers) and convert the residues to `int` for the heatmap. The pictures are the same as the original's; the note after the divisor-5 cells explains why.

In [ ]:
size = 100
divisor = 2
mod = lambda x: x%divisor
a = ndplex(size, dtype=object)
sns.heatmap(mod(a).astype(int))

Increasing `divisor` adds colors and detail.

In [ ]:
size = 100
divisor = 4
mod = lambda x: x%divisor
a = ndplex(size, dtype=object)
sns.heatmap(mod(a).astype(int))

What happens if the divisor is 5? Notice how the triangle pattern is maintained until around column 16 and then appears random? Why?

> **Package note.** This cell keeps `int64` on purpose: the overflow is the puzzle. The `RuntimeWarning` it prints is expected; it names the first node, in generation order, whose value overflowed.

In [ ]:
size = 100
divisor = 5
mod = lambda x: x%divisor
a = ndplex(size,dtype=np.int64)
sns.heatmap(mod(a))

You can correct this by using the `mod` function in the `func` paramenter of the ndplex, rather than in the call to `heatmap`. Otherwise, node values begin to exceed the maximum value of `int64` within the bounds of the graph. Curiously, you do not need to make this correction if `divisor` is a power of 2.

In [ ]:
size = 100
divisor = 5
mod = lambda x: x%divisor
rule = lambda x: mod(x[0] + x[1])
a = ndplex(size,dtype=np.int64, func=rule)
sns.heatmap(a)

> **Package note.** *Why the divisor-5 picture breaks and the divisor-2 and 4 pictures don't* (found in the 2026 audit; *proved*, and checked on this 100 × 100 array). `int64` arithmetic is arithmetic modulo 2<sup>64</sup>, so an overflowed value is still correct modulo 2, 4, 8, … (any power of 2 up to 2<sup>64</sup>) but not modulo 3, 5 or 7. So the `int64` heatmaps for divisors 2 and 4 are right, and the one for 5 is wrong from row + column = 67 on. The check below compares the `int64` residues with the exact ones.

In [ ]:
import warnings

with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)   # int64 overflow, expected here
    wrapped = ndplex(100)                              # int64, overflowed
exact = ndplex(100, dtype=object)

for divisor in (2, 4, 8, 3, 5, 7):
    wrong = int(((wrapped % divisor) != (exact % divisor).astype(int)).sum())
    print(f"divisor {divisor}: {wrong} of 10000 residues wrong in int64")

Let's incorporate digit sums into the `func` parameter. Notice that we get a similar triangle pattern to the above.

In [ ]:
# see Andreas K.'s answer to https://stackoverflow.com/questions/48817036
# for source of sum_digits function below

def sum_digits(num):
    num = str(num)
    if len(num) < 2:
        return int(num)
    else:
        return sum_digits(sum([int(dig) for dig in str(num)]))

size = 100
rule = lambda x: sum_digits(x[0] + x[1])
a = ndplex(size,dtype=np.int64, func=rule)
sns.heatmap(a)

Use colorbrewer to choose other colormaps: http://colorbrewer2.org/#type=diverging&scheme=RdGy&n=9

In [ ]:
sns.heatmap(mod(a),cmap="RdGy")

## Applications

### Geometry

See for background:
* Thomas M. Green, *The Simplex, Duplex and Pascal's Triangles: Relatives of Pascal's Triangle, with Excursions Into Hyperspace* (2015): https://books.google.com/books/about/The_Simplex_Duplex_and_Pascal_s_Triangle.html?id=P0cDswEACAAJ
* https://en.wikipedia.org/wiki/Hypercube
* https://en.wikipedia.org/wiki/Simplex

Another name for Pascal's Triangle is "Simplex Triangle".

In [ ]:
size = 7
n = 1
left_edge = n**np.arange(size) #powers of n
rule = lambda x: n*x[0] + x[1]
#if n==1, this is Pascal's Rule; if n==2, the duplex rule; etc
simplex = ndplex(size, edges=(left_edge,1), func=rule)
simplex.as_tri()

In [ ]:
simplex.as_tri(sums=True)

In [ ]:
plot_ndplex(simplex, as_tri='center', title='Pascal\'s or Simplex Triangle',
            caption='rule = n * x[0] + x[1], left edge = powers of n, and n = 1',
            xlabel=None, ylabel=None)

In [ ]:
size = 7
n = 2
left_edge = n**np.arange(size) #powers of n
rule = lambda x: n * x[0] + x[1]
duplex = ndplex(size, edges=(left_edge,1), func=rule)
duplex.as_tri()

In [ ]:
duplex.as_tri(sums=True)

Row 1 of this triangle are the elements of a line segment (i.e. a 1D duplex): 2 vertices and 1 line segment.

Row 2 of this triangle are the elements of a square (i.e. a 2D duplex): 4 vertices, 4 faces, and 1 square.

Row 3  are the elements of a cube (i.e. a 3D duplex): 8 vertices, 12 edges, 6 faces, and 1 cube.

and so on

In [ ]:
duplex.as_tri()[3]

In [ ]:
plot_ndplex(duplex, as_tri='center', title='Duplex Triangle',
            caption='rule = n * x[0] + x[1], left edge = powers of n, and n = 2',
            xlabel=None, ylabel=None)

...and here is a triplex triangle. If `n` is set to 4, then it's a quadplex triangle.

In [ ]:
size = 7
n = 3
left_edge = n**np.arange(size) #powers of n
rule = lambda x: n * x[0] + x[1]
triplex = ndplex(size, edges=(left_edge,1), func=rule)
triplex.as_tri()

In [ ]:
triplex.as_tri(sums=True)

In [ ]:
plot_ndplex(triplex, as_tri='center', title='Triplex Triangle',
            caption='rule = n * x[0] + x[1], left edge = powers of n, and n = 3',
            xlabel=None, ylabel=None)

Here is a visual comparison of the bases of the the n-plex triangles through n=4:

In [ ]:
size = 20

plexes = [[1, "Simplex, or Pascal\'s Triangle"],
          [2, "Duplex"],
          [3, "Triplex"],
          [4, "Quadplex"]]

y, labels = [],[]
for plex in plexes:
    n = plex[0]
    powers = n**np.arange(size)
    rule = lambda x: n * x[0] + x[1]
    p = ndplex(size, edges=(powers,1), func=rule, dtype=float)
    y.append(p.diag(loc=(size-1,0),move=(-1,1)))
    labels.append(plex[1])

diag_as_hist(y, labels=labels, normed=True)

#### Demonstrating the Triangular Numbers: The Special Case of `func=lambda x:1`

Recall that row 2 of the standard array are the triangular numbers. https://oeis.org/A000217

In [ ]:
ndplex(9)[2]

We can demonstrate that this is true by creating a 2D array with the special case of `func=lambda x:1`.

In [ ]:
p_ones_2D = ndplex(9,edges=2,func=lambda x: 1)
p_ones_2D

Then arrange as a triangle with the `as_tri` function.

In [ ]:
p_ones_2D.as_tri()

and take the cululative sum of each row.

In [ ]:
p_ones_2D.as_tri(sums=True).cumsum(-1)

And recall that row 3 of the standard array are the tetrahedral numbers. https://oeis.org/A000292

In [ ]:
ndplex(5)[3]

We can demonstrate that this is true by creating a 3D array with the special case of `func=lambda x:1`

In [ ]:
p_ones_3D = ndplex(5,edges=3,func=lambda x:1)
p_ones_3D

then arrange as a tetrahedron with the `as_tri` function

In [ ]:
p_ones_3D.as_tri()

If we then take the cumulative sums of by layer, we'll see that we get the tetrahedral numbers.

In [ ]:
p_ones_3D.as_tri(sums=True).cumsum(-1)

We can similarly demostrate that row four of the of the standard array...

In [ ]:
ndplex(5)[4]

...are the 4-simplex (i.e. 4-dimensional triangle) numbers:

In [ ]:
p_ones_4D = ndplex(5,edges=4,func=lambda x:1)
p_ones_4D.as_tri(sums=True).cumsum(-1)

### Probability and Statistics

How Pascal's Triangle is related to the bell curve: https://www.ams.org/publicoutreach/feature-column/fcarc-normal (Tony Phillips, AMS Feature Column, May 2010)

Whereas the base of a Pascal's triangle is a binomial distribution, it approaches the normal distribution as the size increases.

In [ ]:
sizes = [5,10,20,40]

y, labels = [],[]
for size in sizes:
    p = ndplex(size, dtype=float)
    y.append(p.diag(loc=(size-1,0),move=(-1,1)))
    labels.append("size = " + str(size))

diag_as_hist(y, labels=labels, normed=True)

The fit to a bell curve applies even in the case of *non-standard* arrays. Compare different values of  the `edges` parameter.

In [ ]:
size = 20
edges_list = [[(1,3), "big edge"],
              [(1,1), "standard"]]

y, labels = [],[]
for edges in edges_list:
    p = ndplex(size,edges=edges[0],dtype=float)
    y.append(p.diag(loc=(size-1,0),move=(-1,1)))
    labels.append(edges[1])

diag_as_hist(y, labels=labels, normed=False)

In [ ]:
diag_as_hist(y, labels=labels, normed=True)

Compare different values of  the `func` parameter.

In [ ]:
size = 15

func_list = [[lambda x: x[0]+x[1], "standard: x[0]+x[1]"],
             [lambda x: x[0]+x[1]*2, "x[0] + x[1]*2"],
             [lambda x: x[0]*2+x[1], "x[0]*2 + x[1]"]]

y, labels = [],[]
for f in func_list:
    p = ndplex(size, func=f[0])
    y.append(p.diag(loc=(size-1,0),move=(-1,1)))
    labels.append(f[1])

diag_as_hist(y, labels=labels, normed=True)

Compare  ascending with descending edges:

In [ ]:
size = 20

nums = np.arange(size)
nums_desc = np.flip(nums)

edges_list = [[(nums_desc,nums_desc), "descending"],
              [(nums,nums), "ascending"]]

y, labels = [],[]
for edges in edges_list:
    p = ndplex(size,edges=edges[0],dtype=float)
    y.append(p.diag(loc=(size-1,0),move=(-1,1)))
    labels.append(edges[1])

diag_as_hist(y, labels=labels, normed=True)

Even when the edges are a random shuffle, the chart looks mostly the same:

In [ ]:
size = 20

nums = np.arange(size)
avg = np.average(nums)

np.random.shuffle(nums)
rando1 = nums.copy()

np.random.shuffle(nums)
rando2 = nums.copy()

edges_list = [[(avg,avg), "constant"],
              [(rando1,rando2), "random"]]

y, labels = [],[]
for edges in edges_list:
    p = ndplex(size,edges=edges[0],dtype=float)
    y.append(p.diag(loc=(size-1,0),move=(-1,1)))
    labels.append(edges[1])

diag_as_hist(y, labels=labels, normed=True)

In [ ]:
sizes = [5,10,20,40]

y, labels = [],[]
for size in sizes:

    nums = np.arange(size)
    avg = np.average(nums)

    np.random.shuffle(nums)
    rando1 = nums.copy()

    np.random.shuffle(nums)
    rando2 = nums.copy()

    p = ndplex(size,edges=[rando1,rando2],dtype=float)
    y.append(p.diag(loc=(size-1,0),move=(-1,1)))
    labels.append(str(size))

diag_as_hist(y, labels=labels, normed=True)

Compare bases with different slopes.

In [ ]:
size = 50
labels = []
j = ndplex(size,dtype=float)
y = []
for i in range(1,5):
    y.append(j.diag(loc=(size-1,0), move=(-i,1)))
    labels.append(str(i))
diag_as_hist(y, labels=labels)

And here is a goofy example. Under what conditions does the histogram not approach the shape of the bell curve?

In [ ]:
size = 50
func = lambda x: x[0]/(x[1]+1)
p = ndplex(size,func=func, dtype=float)
y = p.diag(loc=(size-1,0),move=(-1,1))
diag_as_hist(y, normed=False)

### Fibonacci Numbers, Cows, and Population Growth

In [ ]:
def fibonaccis(size):
    fibonaccis = [1,1]
    for i in range(2,size):
        fibonaccis.append(fibonaccis[i-1] + fibonaccis[i-2])
    return fibonaccis

You can get the Fibonaccis by building a series of steep diagonals, like this,...

In [ ]:
p=ndplex(9)
p.diag(move=[(-2,1),(1,0)],sum_axis=None)

...and summing each row.

In [ ]:
fibs = p.diag(move=[(-2,1),(1,0)],sum_axis=1)
fibs

In [ ]:
plot_ndplex(p, move=(-1,1), as_tri='center', title='The Fibonaccis from Pascal\'s Triangle',
            caption='diagonal sums = ' + str(fibs),
            xlabel=None, ylabel=None)

Ratio of successive Fibonaccis approaches the Golden Ratio.

In [ ]:
fibs = ndplex(50, dtype=object).diag(move=[(-2,1),(1,0)],sum_axis=1)
fibs[-1]/fibs[-2]

Instead of moving up 2 as in the case of the Fibonaccis above, let's try moving up 3:

In [ ]:
p.diag(move=[(-3,1),(1,0)],sum_axis=1)

which turns out to be the Narayana's cows sequence (https://oeis.org/A000930; see also J.-P. Allouche and T. Johnson, *Narayana's Cows and Delayed Morphisms*, JIM 96: https://oeis.org/A000930/a000930.pdf), a model of population growth. So, how many cows and calves can we expect to have after 20 years?

In [ ]:
year = 20
cow_population = ndplex(50, dtype=object).diag(move=[(-3,1),(1,0)],sum_axis=1)
cow_population[year]

But this neglects the death of the first cows. So how do we account for mortality? Subtract diagonals with an offset equal to the reproductive lifespan.

In [ ]:
lifespan = 6
cow_pop2 = cow_population[lifespan:] - cow_population[:-lifespan]
cow_pop2[year-lifespan]

is how many cows and calves we can expect after 20 years.

Now, what happens when we sum diagonals, starting with the horizontal, then incrementally increase the slope?

In [ ]:
size = 10
d = ndplex(size)
b=np.zeros(size,int)
for i in range(1,size+1):
    b = np.append(b,d.diag(move=[(-i,1),(1,0)],sum_axis=1))
b=np.trim_zeros(b)
b = b.reshape(size,-1) #-1 means the second dimension inferred from other values
b

which can be found here: https://oeis.org/A141539. The first three rows are, respectively, the powers of 2, Fibonaccis, and Narayana's cows sequence.

In [ ]:
# Needs the network: looks the rows up on oeis.org.
oeis(b)

Compare the diagonals. `move = (-1,1)` is the base of Pascal's Triangle and is symmetrical. `move = (-2,1)` is the diagonal summed to get a Fibonnaci number.

In [ ]:
size = 20
p = ndplex(size,edges=2,dtype=float)

z, labels = [], []
for i in range(1,5):
    move = (-i,1)
    y = p.diag(loc=(size-1,0),move=move)
    z.append(y)
    labels.append("move = " + str(move))

diag_as_hist(z, labels=labels, normed=True)

Now let's build a `ndplex` with the Fibonacci numbers on the edges.

In [ ]:
size = 9
f = fibonaccis(size)
fplex = ndplex(size, edges=(f,f))
fplex

In [ ]:
# Needs the network: looks the rows up on oeis.org.
oeis(fplex.as_tri())

Here is a puzzler: We put the fibbonacci's on the first row of an otherwise default ndplex. We then heatmap the array modulo 7. What accounts for the diagonal stripes?

> **Package note.** Built with `dtype=object`, so every node is exact. In the original (`int64`), node values overflowed from row + column = 66 on, and 405 of the 2,500 cells (all in that lower-right corner) showed the wrong residue, so that corner differs from the original picture. The diagonal stripes are genuine: they are clearest in the exact array.

In [ ]:
size = 50
divisor = 7
a = ndplex(size, edges=(fibonaccis(size),1), dtype=object)
sns.heatmap((a%divisor).astype(int))

#### Another way to estimate the Golden Ratio:

In [ ]:
size = 20
func = lambda x: (x[0]+x[1])/x[0]
p2= ndplex(size,edges=(1,range(1,size+1)),func=func,dtype=float)
y = p2.diag(loc=(0,1), move=(1,0))
x = np.arange(size)
sns.lineplot(x=x,y=y)
print(y[-1])

### Moving Costs, Turnover, and Rental Rates

In [ ]:
size = 50  # either number of lease periods or number of rental units

turnover = 0.4 # fraction of tenants per lease period which move out
renewals = 1 - turnover # fraction of tenants which renew their leases

moving_costs = 600
market_rent_rate = 450
renewal_rent_rate = market_rent_rate + moving_costs/12.0
# assumes that rational tenants are willing to renew if the new rental rate
# is less than or equal to the market rent plus prorated moving costs

f = lambda x: x[1]*renewals + x[0]*turnover #weighted average rental rate
p_rates = ndplex(size,edges=[renewal_rent_rate, market_rent_rate],
                 func=f,dtype=float)
rental_rates = p_rates.diag(loc=[size-1,0], move=[-1,1])

f2 = lambda x: x[1]*renewals/turnover + x[0] #weighted average renewed
p_occs = ndplex(size, func=f2, dtype=float)
weights = p_occs.diag(loc=[size-1,0], move=[-1,1])

# fill in the gaps
newx = np.linspace(min(rental_rates),max(rental_rates),size)
rental_rates, weights  = zip(*sorted(zip(rental_rates, weights)))
newy = np.interp(newx, xp=rental_rates, fp=weights)

diag_as_hist(newy, x_values=newx, normed=True, labels=['distribution of rates'])
avg_rental_rate = np.average(rental_rates,weights=weights).round(2)
print("Average rental rate = $" + str(avg_rental_rate))

### Searching for Patterns among the Primes

some ideas
* func: left divisible by right
* func: left modulo right
* primes triangle analog to duplex triangle
* first and second differences
* varying `arg_locs`

> **Package note.** The original downloaded the first 1000 primes from a GitHub gist. This cell computes them with `primesfrom2to`, which the original defined in the same cell, so the section runs offline. The 1000th prime is 7919.

In [ ]:
# https://stackoverflow.com/questions/2068372/fastest-way-to-list-all-primes-below-n/3035188#3035188
def primesfrom2to(n):
    """ Input n>=6, Returns a array of primes, 2 <= p < n """
    sieve = np.ones(n//3 + (n%6==2), dtype=bool)
    for i in range(1,int(n**0.5)//3+1):
        if sieve[i]:
            k=3*i+1|1
            sieve[       k*k//3     ::2*k] = False
            sieve[k*(k-2*(i&1)+4)//3::2*k] = False
    return np.r_[2,3,((3*np.nonzero(sieve)[0][1:]+1)|1)]

#primes, first 1000 (computed; the original downloaded them from a gist)
primes = primesfrom2to(8000)[:1000].tolist()
print(len(primes), "primes, the last is", primes[-1])

#### edges=(primes, constant, ...)

Let's put the primes in row 0. Row 1 is interesting in that no numbers are prime: see https://oeis.org/A138981

In [ ]:
size = 8
p = primes[:size]
ndplex(size, edges=(1, p))

Set the left edge to 2 so that both edges share a common origin.

In [ ]:
ndplex(size, edges=(2, p))

...or 1 if you assume the primes start with 1.

In [ ]:
p = [1] + list(primes[:size-1])
arg_locs = [[-1,0],[0,-1]]
prime_matrix = ndplex(size, edges=(1, p), arg_locs=arg_locs)
prime_matrix

In [ ]:
# Needs the network: looks the rows up on oeis.org.
oeis(prime_matrix)

In [ ]:
prime_matrix.diag()

In [ ]:
prime_matrix.as_tri(sums=True)

#### edges=(primes, primes, ...)

In [ ]:
size = 9
p = primes[:size]
prime_matrix = ndplex(size, edges=(p, p))
prime_matrix

In [ ]:
prime_matrix.diag()

In [ ]:
prime_matrix.as_tri(sums=True)

Now lets start the primes with 1 and display the main diagonal.

In [ ]:
p = [1] + list(primes[:size - 1])
prime_matrix = ndplex(size, edges=(p, p))
prime_matrix

In [ ]:
prime_matrix.diag()

In [ ]:
prime_matrix.as_tri(sums=True)

In [ ]:
prime_matrix[1]

#### edges=(primes, nonconstant, ...)

In [ ]:
size = 9
p = primes[:size]
c = np.arange(size)+2
prime_matrix2 = ndplex(size, edges=(c,p))
prime_matrix2

In [ ]:
size = 9
p = [1] + list(primes[:size-1])
c = np.arange(size)+1
prime_matrix1 = ndplex(size, edges=(c,p))
prime_matrix1

Taking the difference between the above two arrays:

In [ ]:
prime_matrix_diff = prime_matrix2 - prime_matrix1
prime_matrix_diff

Row 0 is maybe https://oeis.org/A075526 and column 2 is maybe the lazy caterer's sequence: https://oeis.org/A000124

In [ ]:
prime_matrix_diff.diagonal()

`prime_matrix_diff` can also be defined this way:

In [ ]:
size = 9

p2 = np.asarray(primes[:size])
p1 = np.asarray([1] + list(primes[:size-1]))
p_diff = p2 - p1

prime_matrix_diff2 = ndplex(size, edges=(1,p_diff))
prime_matrix_diff2

In [ ]:
# Needs the network: looks the rows up on oeis.org.
oeis(prime_matrix_diff2,trim=True)

Alien invasion

> **Package note.** The original used `dtype=np.float128`, which does not exist on Windows or on Apple-silicon Macs. Integers don't fit `int64` here either: the Fibonacci numbers on the top edge pass the `int64` maximum from the 93rd on, and the package raises `OverflowError` when an edge doesn't fit the dtype. So this cell uses `dtype=object` (exact). `primesfrom2to(...).tolist()` gives Python integers.
> >
> > The picture differs from the original's in its six rightmost columns. `float128` holds integers exactly only up to 2<sup>64</sup>, and the Fibonacci numbers on the top edge pass that from column 93 on, so some of them were stored with the wrong parity. In the original, 369 interior cells, all in columns 94–99, were wrong (*run* on Linux, comparing the original's `float128` array with this exact one).

In [ ]:
def fibonaccis(size):
    fibonaccis = [1,1]
    for i in range(2,size):
        fibonaccis.append(fibonaccis[i-1] + fibonaccis[i-2])
    return fibonaccis

size = 100
primes = primesfrom2to(1000000).tolist()
p = [1] + primes[:size - 1]
f = fibonaccis(size)
rule = lambda x: (x[0] + x[1])%2
dingbat = ndplex(size, edges=(p,f),func=rule, dtype=object)
sns.heatmap(dingbat[1:,1:].astype(int))

#### `func` parameter

In [ ]:
size = 7
#p = [1] + list(primes[:size-1])
p = primes[:size]
f = lambda x: 2*x[0] + x[1] #duplex rule
primes_2_duplex = ndplex(size, edges=(p, 2), func=f)
primes_2_duplex

In [ ]:
size = 7
p = [1] + list(primes[:size-1])
#p = primes[:size]
f = lambda x: 2*x[0] + x[1] #duplex rule
ndplex(size, edges=(p, 1), func=f)

In [ ]:
size = 7
p = [1] + list(primes[:size-1])
c = np.arange(size)+1
f = lambda x: 2*x[0] + x[1] #the duplex rule
ndplex(size, edges=(p, c), func=f)

#### More than 2 dimensions.

In [ ]:
size = 10
p = [1] + list(primes[:size-1])
prime_xD = ndplex(size, (p,1,1))
prime_xD.diag()

In [ ]:
prime_xD.as_tri(sums=True)

#### Adjacent diagonals ratios

In [ ]:
def adj_ratios(g):
    ratios = g[1:]/g[:-1]
    print("first 5 ratios =", ratios[:5])
    print("last = ",ratios[-1])
    sns.lineplot(x=np.arange(len(g)-1), y=ratios)

In [ ]:
size = 100
#p = [1] + list(primes[:size-1])
p = primes[:size]
prime_matrix_big = ndplex(size, edges=(p,p), dtype=float)

d = prime_matrix_big.diag(loc=(0,0), move=(1,1))
adj_ratios(d)

#### Proximity to the nearest prime number

> **Package note.** Changed (audit item N8). The original searched for the nearest prime among the primes *below* the array's maximum, so nodes above the largest such prime were measured against it instead of against the next prime up. By Bertrand's postulate there is always a prime between n and 2n, so the primes below twice the maximum include the nearest prime to every node. The printed array differs from the original's in one place (see the next note).

In [ ]:
size = 10

def evens(size):
    return np.arange(1,size+1)*2

e = evens(size)

func = lambda x: x[0] + x[1] - 1
sam = ndplex(size,edges=(e,e),func=func)

shape = sam.shape
sam = sam.reshape(size**2,1)

# all primes below twice the maximum (was: below the maximum)
primes = primesfrom2to(2 * sam.max())

mod = lambda x: np.abs(primes-x).min()
prox = np.asarray(list(map(mod, sam)))

prox = prox.reshape(shape)

prox

> **Package note.** Compared with the original output, only the maximum node changes: 223 653, at row 9, column 9, is 14 from the nearest prime (223 667), not 16 (from 223 637, the largest prime below the maximum). The 16s at (3, 9) and (9, 3) are unchanged: those nodes are 1343, and the nearest prime to 1343 is 1327.

In [ ]:
sns.heatmap(prox)